<a href="https://colab.research.google.com/github/kimgayeon430/skala-LangChain/blob/main/init_chat_model_ipynb%EC%9D%98_%EC%8B%A4%ED%96%89_%EA%B2%B0%EA%B3%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 생성형 AI 서비스 개발의 이해 활용 (LangChain)

## 환경설정

### API Key

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다:", os.environ["GOOGLE_API_KEY"][:5]+"*****")

.env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다: sk-pr*****
.env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다: AQ.Ab*****


### Install package

In [ ]:
#여기서 설치 하지 않음. 아래에 별도 설치
#!pip install langchain-openai
#!pip install langchain-google-genai

## OpenAI, Google GenAI의 전용 library 사용

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

city = "제주도"
prompt = f"너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.7,
)

# 텍스트 응답 추출 및 수동 후처리
raw_text = response.choices[0].message.content
landmarks = [item.strip() for item in raw_text.split(",")]

print("OpenAI 결과:", landmarks)

OpenAI 결과: ['한라산', '성산 일출봉', '만장굴']


In [ ]:
import os
from google import genai

client = genai.Client()

city = "제주도"
prompt = f"너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘."

chat = client.chats.create(model="gemini-3.5-flash-lite")

response = chat.send_message(prompt)

# 텍스트 응답 추출 및 수동 후처리
raw_text = response.text
landmarks = [item.strip() for item in raw_text.split(",")]

print("Google 결과:", landmarks)

Google 결과: ['안녕하세요! 제주도 여행 가이드입니다. 추천해 드리는 제주도의 대표 관광 명소 3곳입니다.\n\n성산일출봉', '한라산국립공원', '우도']


## init_chat_model 사용

In [ ]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain.chat_models import init_chat_model

## integration 용 OpenAI 패키지 설치 전

In [ ]:
# 해당 세션을 돌리기 위해 langchain-openai 패키지가 뒷단에서 필요 -> 설치전 실행 -> 오류남
# 1. 컴포넌트 선언
parser = CommaSeparatedListOutputParser()

template = """너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘.
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["city"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

model="openai:gpt-4o-mini"
#model="google_genai:gemini-2.5-flash-lite"

# 2. 모델 선택 (OpenAI ↔ Gemini 간 교체는 오직 model, model_provider만 변경)
llm = init_chat_model(model, temperature=0.7)

# 3. 파이프라인 조립 (LCEL)
chain = prompt | llm | parser

# 4. 실행
result = chain.invoke({"city": "제주도"})
print("사용한 model:", model)
print("LangChain 결과:", result)

ImportError: Initializing ChatOpenAI requires the langchain-openai package. Please install it with `pip install langchain-openai`

In [ ]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


## integration 용 OpenAI 패키지 설치 후

In [ ]:
# 1. 컴포넌트 선언
parser = CommaSeparatedListOutputParser()

template = """너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘.
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["city"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

model="openai:gpt-4o-mini"
#model="google_genai:gemini-2.5-flash-lite"

# 2. 모델 선택 (OpenAI ↔ Gemini 간 교체는 오직 model, model_provider만 변경)
llm = init_chat_model(model, temperature=0.7)

# 3. 파이프라인 조립 (LCEL)
chain = prompt | llm | parser

# 4. 실행
result = chain.invoke({"city": "제주도"})
print("사용한 model:", model)
print("LangChain 결과:", result)

사용한 model: openai:gpt-4o-mini
LangChain 결과: ['한라산', '성산일출봉', '만장굴']


## integration 용 Google 패키지 설치 전

In [ ]:
# 1. 컴포넌트 선언
parser = CommaSeparatedListOutputParser()

template = """너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘.
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["city"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

#model="openai:gpt-4o-mini"  -> 이 부분만 openai 셀과 다름
model="google_genai:gemini-2.5-flash-lite"


# 2. 모델 선택 (OpenAI ↔ Gemini 간 교체는 오직 model, model_provider만 변경)
llm = init_chat_model(model, temperature=0.7)

# 3. 파이프라인 조립 (LCEL)
chain = prompt | llm | parser

# 4. 실행
result = chain.invoke({"city": "제주도"})
print("사용한 model:", model)
print("LangChain 결과:", result)

ImportError: Initializing ChatGoogleGenerativeAI requires the langchain-google-genai package. Please install it with `pip install langchain-google-genai`

In [ ]:
!pip install langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## integration 용 Google 패키지 설치 후

In [ ]:
# 1. 컴포넌트 선언
parser = CommaSeparatedListOutputParser()

template = """너는 여행 가이드야. {city}의 대표 관광 명소 3곳을 쉼표(,)로 구분해서 이름만 나열해줘.
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["city"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

#model="openai:gpt-4o-mini"
model="google_genai:gemini-3.5-flash-lite"


# 2. 모델 선택 (OpenAI ↔ Gemini 간 교체는 오직 model, model_provider만 변경)
llm = init_chat_model(model, temperature=0.7)

# 3. 파이프라인 조립 (LCEL)
chain = prompt | llm | parser

# 4. 실행
result = chain.invoke({"city": "제주도"})
print("사용한 model:", model)
print("LangChain 결과:", result)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


사용한 model: google_genai:gemini-3.5-flash-lite
LangChain 결과: ['성산일출봉', '한라산', '만장굴']
